In [7]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

# 1. LOAD DATA
df = pd.read_csv(r'C:\Users\Soumya_SRB\Desktop\Netflix-Content-Clustering-Engine\NetflixSimple.csv')

# 2. HANDLE MISSING VALUES 
df.fillna('Unknown', inplace=True)

# 3. NLP: CLEAN AND CREATE "BAG OF CONTENT"
def clean_text(text):
    text = str(text).lower()
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    return text
df['text_features'] = (df['director'] + ' ' + df['cast'] + ' ' + 
                      df['listed_in'] + ' ' + df['description']).apply(clean_text)

# 4. VECTORIZATION & DIMENSIONALITY REDUCTION
tfidf = TfidfVectorizer(stop_words='english', max_features=3000)
tfidf_matrix = tfidf.fit_transform(df['text_features'])
svd = TruncatedSVD(n_components=50, random_state=42)
matrix_reduced = svd.fit_transform(tfidf_matrix)

# 5. CLUSTERING & VALIDATION
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(matrix_reduced)

# 6. RECOMMENDATION ENGINE
cosine_sim = cosine_similarity(matrix_reduced)
def get_recommendations(title, df=df, cosine_sim=cosine_sim):
    try:
        idx = df[df['title'].str.lower() == title.lower()].index[0]
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        movie_indices = [i[0] for i in sim_scores[1:6]]
        return df[['title', 'listed_in']].iloc[movie_indices]
    except:
        return "Title not found. Please check the spelling."

# --- RESULTS ---
print(f"Project Status: Success")
print(f"Validation Score: {silhouette_score(matrix_reduced, df['cluster']):.4f}")
print("\nTop 5 Recommendations for '3 Idiots':")
print(get_recommendations("3 Idiots"))

Project Status: Success
Validation Score: 0.1221

Top 5 Recommendations for '3 Idiots':
                      title                               listed_in
1758        Dil Dhadakne Do  Comedies, Dramas, International Movies
2006  English Babu Desi Mem  Comedies, Dramas, International Movies
4872                     PK  Comedies, Dramas, International Movies
4846        Phir Hera Pheri          Comedies, International Movies
1682   Deewana Main Deewana  Comedies, Dramas, International Movies
